In [1]:
import sys
sys.path.append('/n/home06/drooryck/codeswitching-llms')
from pathlib import Path
from july_aug_exp.src.metrics import Metrics
from july_aug_exp.src.dataset_manager import DatasetManager
from july_aug_exp.src.model_config import ModelConfig, SlurmConfig
from july_aug_exp.src.experiment import Experiment
from july_aug_exp.src.plotting import BilingualPlotter

KeyboardInterrupt: 

In [ ]:
from july_aug_exp.src.metrics import Metrics
from july_aug_exp.src.dataset_manager import DatasetManager
from july_aug_exp.src.model_config import ModelConfig, SlurmConfig
from july_aug_exp.src.experiment import Experiment
from july_aug_exp.src.plotting import BilingualPlotter

repo_root = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/")

output_dir = repo_root / "results" / "sep15.9"
output_dir.mkdir(parents=True, exist_ok=True)

# dataclass for setting transformers-lib params
config = ModelConfig.load(repo_root / "configs" / "default_model.json")
# dataclass for what gpu to run on, how long to run etc
slurm_config = SlurmConfig.load(repo_root / "configs" / "slurm_default.json")
slurm_config.job_name = "sep22.1exp"

# handles creating sentences from lexicon, custom tokenizer logic
data_manager = DatasetManager("data", config)
# handles calculating metrics on sentences
metrics = Metrics("data/lexicon_new.json")
 # run_single method does the lifting and calls the other methods,
experiment = Experiment(config, data_manager, metrics, output_dir)
# training, inference, and collecting results happens here


config.save(output_dir / "model_config.json")
slurm_config.save(output_dir / "slurm_config.json")

props = [ # proportions of french sentences used in training data in each run
    0.0, 0.001, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03, 0.04, 0.05, 0.075,
    0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7,
    0.75, 0.8, 0.85, 0.9,
    0.925, 0.95, 0.96, 0.97, 0.975, 0.98, 0.985, 0.99, 0.995, 0.999, 1.0
]
runs = [1, 2, 3] # the ids here are random seeds for data splitting, and slurm IDs

job_ids = experiment.submit_to_slurm( # as easy as that (x
    props=props,
    runs=runs,
    slurm_config=slurm_config,
    eval_prop=0.1
)

## run cluster job

In [4]:
# Submit jobs to SLURM
print("Submitting jobs to SLURM...")
job_ids = experiment.submit_to_slurm(
    props=props,
    runs=runs,
    slurm_config=slurm_config,
    eval_prop=0.1
)

print(f"\nJobs submitted successfully!")
print(f"Job IDs: {job_ids}")
print(f"Results will be saved in: {output_dir}")
print(f"SLURM logs will be in: {logs_dir}")

2025-09-15 14:49:24,411 |     INFO | Submitting 39 jobs to SLURM...


Submitting jobs to SLURM...


2025-09-15 14:49:24,798 |     INFO | Submitted job array 35731868



Jobs submitted successfully!
Job IDs: ['35731868']
Results will be saved in: /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9
SLURM logs will be in: /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/logs


## run sweep local

In [ ]:
def collect_results(self, run_dirs: List[Path]) -> pd.DataFrame:
    """Collect metrics from multiple experiment runs.

    Args:
        run_dirs: List of experiment output directories

    Returns:
        DataFrame with aggregated results
    """
    results = []

    for run_dir in run_dirs:
        metrics_file = run_dir / "metrics.json"
        if not metrics_file.exists():
            self.logger.warning(f"No metrics found in {run_dir}")
            continue

        with open(metrics_file, 'r') as f:
            metrics = json.load(f)

        # Extract run parameters from directory name
        dir_name = run_dir.name
        if dir_name.startswith("p") and "_run" in dir_name:
            parts = dir_name.split("_run")
            prop = int(parts[0][1:]) / 100.0
            run_id = int(parts[1])
        else:
            prop = None
            run_id = None

        result = {
            'prop': prop,
            'run_id': run_id,
            'run_dir': str(run_dir),
            **metrics
        }
        results.append(result)

    return pd.DataFrame(results)

In [ ]:
# results_dirs = experiment.run_sweep(props=props, runs=runs, eval_prop=0.05)

# results_df = experiment.collect_results(results_dirs)
# experiment.save_summary(results_df)
# experiment.create_plots(results_df)

In [ ]:
    # def save_summary(self, results_df: pd.DataFrame,
    #                 output_path: Optional[Path] = None) -> None:
    #     """Save experiment summary.

    #     Args:
    #         results_df: Results dataframe
    #         output_path: Path to save summary (defaults to output_dir/summary.csv)
    #     """
    #     if output_path is None:
    #         output_path = self.output_dir / "summary.csv"

    #     results_df.to_csv(output_path, index=False)
    #     self.logger.info(f"Summary saved to {output_path}")

    # def create_plots(self, results_df: pd.DataFrame) -> None:
    #     """Create visualization plots from results.

    #     Args:
    #         results_df: Results dataframe with metrics computed on model outputs
    #     """
    #     plotter = BilingualPlotter(results_df, self.output_dir / "plots")
    #     plotter.create_all_plots()


## plot

In [10]:
from pathlib import Path

output_dir = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/")
run_dirs = list(output_dir.glob("p*_run*"))
results_df = experiment.collect_results(run_dirs)
experiment.save_summary(results_df)

plotter = BilingualPlotter(results_df, output_dir / "plots")
plotter.print_metrics_summary()
plotter.create_all_plots()

2025-09-15 15:22:42,530 |  WARNING | No metrics found in /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/p995_run3
2025-09-15 15:22:42,552 |  WARNING | No metrics found in /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/p999_run3
2025-09-15 15:22:42,571 |     INFO | Summary saved to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/summary.csv
2025-09-15 15:22:42,594 |     INFO | Creating exact match plot...



Available metrics:
['prop', 'run_id', 'run_dir', 'fr_exact', 'fr_avg_fr', 'fr_avg_nl', 'fr_part_final', 'fr_verb_lang', 'fr_verb_choice', 'fr_aux_form', 'fr_det_lang', 'fr_det_agreement', 'nl_exact', 'nl_avg_fr', 'nl_avg_nl', 'nl_part_final', 'nl_verb_lang', 'nl_verb_choice', 'nl_aux_form', 'nl_det_lang', 'nl_det_agreement', 'overall_exact']

Runs per proportion:
prop
0.00     1
0.01     1
0.05     1
0.10     1
0.15     1
0.20     1
0.25     1
0.30     1
0.40     1
0.50     1
0.75     1
1.00     1
1.50     1
2.00     1
2.50     1
3.00     1
3.50     1
4.00     1
4.50     1
5.00     1
5.50     1
6.00     1
6.50     1
7.00     1
7.50     1
8.00     1
8.50     1
9.00     1
9.25     1
9.50     1
9.60     1
9.70     1
9.75     1
9.80     1
9.85     1
9.90     1
10.00    1
dtype: int64

Summary statistics:
            prop  run_id   fr_exact  fr_avg_fr  fr_avg_nl  fr_part_final  \
count  37.000000    37.0  37.000000  37.000000  37.000000           37.0   
mean    4.731351     3.0   0.935054

2025-09-15 15:22:42,992 |     INFO | Saved plot to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/plots/exact_match.png
2025-09-15 15:22:42,993 |     INFO | Creating word order plot...
2025-09-15 15:22:43,439 |     INFO | Saved plot to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/plots/word_order.png
2025-09-15 15:22:43,441 |     INFO | Creating French token distribution plot...
2025-09-15 15:22:43,935 |     INFO | Saved plot to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/plots/french_token_dist.png
2025-09-15 15:22:43,936 |     INFO | Creating Dutch token distribution plot...
2025-09-15 15:22:44,351 |     INFO | Saved plot to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/plots/dutch_token_dist.png
2025-09-15 15:22:44,353 |     INFO | Creating verb metrics plots...
2025-09-15 15:22:44,852 |     INFO | Saved plot to /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15.9/plots/french_verb_me

## generate metrics

In [9]:
# Initialize metrics calculator with lexicon
lexicon_path = Path("/n/home06/drooryck/codeswitching-llms/july_aug_exp/data/lexicon_new.json")
metrics_calculator = Metrics(lexicon_path)

# Process both directories
dirs = [
    "/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15/p99_run1",
    "/n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15/p99_run2"
]

for dir_path in dirs:
    dir_path = Path(dir_path)
    pred_file = dir_path / "predictions.csv"
    metrics_file = dir_path / "metrics.json"
    
    if pred_file.exists():
        # Read predictions
        df = pd.read_csv(pred_file)
        
        # Convert to list of dicts format expected by compute_all_metrics
        predictions = df.to_dict('records')
        
        # Calculate metrics using existing implementation
        metrics = metrics_calculator.compute_all_metrics(predictions)
        
        # Save metrics
        metrics_calculator.save_metrics(metrics, metrics_file)
        print(f"Generated metrics for {dir_path}")
    else:
        print(f"No predictions.csv found in {dir_path}")

Generated metrics for /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15/p99_run1
Generated metrics for /n/home06/drooryck/codeswitching-llms/july_aug_exp/results/sep15/p99_run2
